In [101]:
import pandas as pd

METADATA_FILE = (
    "../data/raw/"
    "GTEx_Analysis_v8_Annotations_SampleAttributesDS.txt"
)

controls = [
    "PT-NPJ7",
    "PT-P44H",
    "PT-Q2AG",
    "PT-QVJO",
    "PT-R55F",
    "PT-RN5K",
    "PT-RU72",
    "PT-UTHO",
    "PT-WVLH",
    "PT-Y8DK",
]

# ------------------------------------------------------------
# Load only the columns we need
# ------------------------------------------------------------

metadata = pd.read_csv(
    METADATA_FILE,
    sep="\t",
    usecols=["SAMPID", "SMTS", "SMTSD"],
    dtype=str
)

# ------------------------------------------------------------
# Print GTEx samples containing the donor code
# ------------------------------------------------------------

print("=" * 70)
print("SEARCHING GTEx METADATA")
print("=" * 70)

for case in controls:

    donor = case.replace("PT-", "")

    # Search anywhere in SAMPID
    matches = metadata[
        metadata["SAMPID"]
        .str.contains(
            donor,
            case=False,
            na=False
        )
    ]

    print("\n" + "-" * 70)
    print(case)
    print("GTEx donor search:", donor)
    print("Matches:", len(matches))

    if len(matches) > 0:

        print(
            matches[
                ["SAMPID", "SMTS", "SMTSD"]
            ].to_string(index=False)
        )

    else:

        print("NO MATCH")

SEARCHING GTEx METADATA

----------------------------------------------------------------------
PT-NPJ7
GTEx donor search: NPJ7
Matches: 35
                      SAMPID         SMTS                                     SMTSD
     GTEX-NPJ7-0006-SM-3GACR        Blood                               Whole Blood
     GTEX-NPJ7-0008-SM-4E3JS         Skin              Cells - Cultured fibroblasts
     GTEX-NPJ7-0009-SM-2BWYS        Blood                               Whole Blood
     GTEX-NPJ7-0009-SM-2XJZQ        Blood                               Whole Blood
     GTEX-NPJ7-0009-SM-3USRM        Blood                               Whole Blood
     GTEX-NPJ7-0009-SM-5JK4M        Blood                               Whole Blood
GTEX-NPJ7-0011-R10A-SM-2I3E5        Brain              Brain - Frontal Cortex (BA9)
GTEX-NPJ7-0011-R10A-SM-2IZHU        Brain              Brain - Frontal Cortex (BA9)
GTEX-NPJ7-0011-R10a-SM-AHZ7T        Brain              Brain - Frontal Cortex (BA9)
GTEX-NPJ7-0011-R11A-

In [104]:
import pandas as pd
from pathlib import Path

# ============================================================
# FILE
# ============================================================

gtex_file = Path(
    "../data/raw/"
    "GTEx_Analysis_2017-06-05_v8_RNASeQCv1.1.9_gene_reads.gct"
)

output_file = Path(
    "../data/processed/"
    "GTEx_7_normal_frontal_cortex_raw_counts.csv"
)

# ============================================================
# CONTROL DONORS
# ============================================================

control_donors = [
    "GTEX-NPJ7",
    "GTEX-P44H",
    "GTEX-Q2AG",
    "GTEX-QVJO",
    "GTEX-R55F",
    "GTEX-RU72",
    "GTEX-UTHO",
    "GTEX-WVLH",
    "GTEX-Y8DK",
]

# ============================================================
# READ GCT HEADER ONLY
# ============================================================

print("=" * 70)
print("READING GTEx GCT HEADER")
print("=" * 70)

header = pd.read_csv(
    gtex_file,
    sep="\t",
    skiprows=2,
    nrows=0
)

all_columns = header.columns.tolist()

print("Total columns:", len(all_columns))

# ============================================================
# FIND AVAILABLE FRONTAL CORTEX SAMPLES
# ============================================================

selected_samples = {}

for donor in control_donors:

    donor_samples = [
        col for col in all_columns
        if col.upper().startswith(donor.upper() + "-")
    ]

    frontal_samples = [
        col for col in donor_samples
        if "-0011-R10A-" in col.upper()
    ]

    # Remove special INPUT / SUP / ELUATE samples
    frontal_samples = [
        col for col in frontal_samples
        if not any(
            x in col.upper()
            for x in ["INPUT", "SUP", "ELUATE"]
        )
    ]

    print(f"\n{donor}")
    print("Frontal cortex samples found:", len(frontal_samples))

    if frontal_samples:
        selected_samples[donor] = frontal_samples[0]
        print("Selected:", frontal_samples[0])
    else:
        print("No matching sample in GCT — skipping")

# ============================================================
# SHOW FINAL CONTROL SET
# ============================================================

print("\n" + "=" * 70)
print("FINAL AVAILABLE CONTROLS")
print("=" * 70)

for donor, sample in selected_samples.items():
    print(f"{donor:12s} -> {sample}")

print("\nNumber of controls:", len(selected_samples))

if len(selected_samples) == 0:
    raise ValueError("No control samples were found.")

# ============================================================
# LOAD ONLY THE AVAILABLE CONTROLS
# ============================================================

keep_cols = [
    "Name",
    "Description"
] + list(selected_samples.values())

print("\n" + "=" * 70)
print("LOADING CONTROL COUNTS")
print("=" * 70)

print("Columns being loaded:", len(keep_cols))
print("This may take a while...")

controls = pd.read_csv(
    gtex_file,
    sep="\t",
    skiprows=2,
    usecols=keep_cols
)

# ============================================================
# CLEAN ENSEMBL IDs
# ============================================================

controls["Name"] = (
    controls["Name"]
    .astype(str)
    .str.split(".")
    .str[0]
)

# Remove duplicate Ensembl IDs
controls = controls.drop_duplicates(
    subset="Name",
    keep="first"
)

# ============================================================
# RENAME SAMPLE COLUMNS TO DONOR IDs
# ============================================================

rename_map = {
    sample: donor
    for donor, sample in selected_samples.items()
}

controls = controls.rename(
    columns=rename_map
)

# ============================================================
# SAVE
# ============================================================

output_file.parent.mkdir(
    parents=True,
    exist_ok=True
)

controls.to_csv(
    output_file,
    index=False
)

# ============================================================
# FINAL OUTPUT
# ============================================================

print("\n" + "=" * 70)
print("DONE")
print("=" * 70)

print("Shape:", controls.shape)

print("\nColumns:")
print(controls.columns.tolist())

print("\nControl donors:")
print(list(selected_samples.keys()))

print("\nSaved to:")
print(output_file)

READING GTEx GCT HEADER
Total columns: 17384

GTEX-NPJ7
Frontal cortex samples found: 1
Selected: GTEX-NPJ7-0011-R10A-SM-2I3E5

GTEX-P44H
Frontal cortex samples found: 1
Selected: GTEX-P44H-0011-R10A-SM-2XCEK

GTEX-Q2AG
Frontal cortex samples found: 1
Selected: GTEX-Q2AG-0011-R10A-SM-2HMLA

GTEX-QVJO
Frontal cortex samples found: 1
Selected: GTEX-QVJO-0011-R10A-SM-2S1QJ

GTEX-R55F
Frontal cortex samples found: 0
No matching sample in GCT — skipping

GTEX-RU72
Frontal cortex samples found: 0
No matching sample in GCT — skipping

GTEX-UTHO
Frontal cortex samples found: 1
Selected: GTEX-UTHO-0011-R10A-SM-3GIJQ

GTEX-WVLH
Frontal cortex samples found: 1
Selected: GTEX-WVLH-0011-R10A-SM-3MJFM

GTEX-Y8DK
Frontal cortex samples found: 1
Selected: GTEX-Y8DK-0011-R10A-SM-4SOK1

FINAL AVAILABLE CONTROLS
GTEX-NPJ7    -> GTEX-NPJ7-0011-R10A-SM-2I3E5
GTEX-P44H    -> GTEX-P44H-0011-R10A-SM-2XCEK
GTEX-Q2AG    -> GTEX-Q2AG-0011-R10A-SM-2HMLA
GTEX-QVJO    -> GTEX-QVJO-0011-R10A-SM-2S1QJ
GTEX-UTHO    ->